# YOLOS-Small (COCO) — DIMER object detection and bounded detection fine-tuning (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/yolos-detection-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/yolos-detection-pipeline/blob/main/tutorials/yolos_detection_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-hustvl%2Fyolos--small-ffcc4d?style=flat)](https://huggingface.co/hustvl/yolos-small) [![Upstream](https://img.shields.io/badge/Upstream-hustvl%2FYOLOS-181717?style=flat&logo=github&logoColor=white)](https://github.com/hustvl/YOLOS) [![arXiv](https://img.shields.io/badge/arXiv-2106.00666-b31b1b.svg)](https://arxiv.org/abs/2106.00666) [![License](https://img.shields.io/badge/License-Apache--2.0-green.svg)](https://github.com/kurtvalcorza/yolos-detection-pipeline/blob/main/LICENSE)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.1 — **standalone** (§4)  
**Capability:** object detection over the COCO classes with YOLOS-Small, a plain Vision Transformer trained as a detector, and a bounded detection fine-tune that re-heads YOLOS onto your own class vocabulary, evaluates it against a held-out split with COCO-style average precision, and exports a reloadable SafeTensors adapter

**This notebook is standalone.** It carries the repository's package (2 modules under `src/yolos_detection_pipeline/`, at revision `128b25d4d0cd`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `3d8f7130d3ce4907cb206fe1c8485dc8fe8703de` (~123 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned checkpoint, runs COCO detection on a drawn scene, validates the 40-image sign dataset, splits it into training and held-out parts, measures the pre-adaptation baseline, **runs the bounded fine-tune**, re-evaluates on the held-out split, detects on unseen images, exports the adapter, reloads it onto a fresh base model to verify the detections, and writes machine-readable outputs with provenance. Nothing is skipped behind a default-off flag, and no clone or DIMER worker is required (NOTEBOOK_SPEC 2.1 §5, RUN7, FT2).

**Bring Your Own Data:** Two optional BYOD branches are included, and both are off by default (`USE_BYOD_IMAGE = False`, `USE_BYOD_DATASET = False`). `USE_BYOD_IMAGE` runs your own image through the same validation, detection and evaluation-report stages as the sample scene. `USE_BYOD_DATASET` takes your own labelled detection records through the full adaptation workflow — validate, split, baseline, fine-tune, evaluate, export and reload — under NOTEBOOK_SPEC 2.1 DAT14. Set `BYOD_IMAGE_PATH` or `BYOD_DATASET_DIR` to read from a location without an upload dialog (EXE2).

YOLOS (You Only Look at One Sequence) Small (`hustvl/yolos-small`) is a plain Vision Transformer (ViT-S/16) turned into a detector with as few changes as possible: 100 learnable **detection tokens** are appended to the image's patch tokens, the 12-layer encoder processes them together, and two small MLP heads read one box and one class distribution from each detection token. It is trained with the same set-prediction loss as DETR, so there is no convolutional backbone, no decoder, no anchor box and no non-maximum suppression (NMS). At inference the processor resizes the image so its shorter side is 800 px, and every detection token whose best class probability reaches a caller-owned threshold is returned as an xyxy box in input pixels.

**The default path really adapts the model:** it re-heads YOLOS onto a three-class traffic sign vocabulary that does not exist in COCO (`stop-sign`, `yield-sign`, `speed-limit-sign`), measures a pre-adaptation baseline, runs a bounded fine-tune with the patch embedding and the first 8 encoder layers frozen, scores the result on a held-out split with COCO-style average precision (AP@[.50:.95] and AP50), runs the adapted model on unseen images, exports the changed tensors as a SafeTensors adapter, and reloads that adapter onto a fresh copy of the verified base model to check that it reproduces the same detections. Every number you see is measured in this notebook runtime.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable upstream model revision; run COCO detection on a drawn scene and score per-object `box_iou`; probe the detector with a blank and a noise image; build and validate a labelled detection dataset over a new three-class sign vocabulary; split it and measure a pre-adaptation baseline; run a bounded fine-tune with the DETR set-prediction loss YOLOS is trained with (Hungarian matching, cross-entropy, L1 and generalised IoU); score the adapted model on the held-out split with COCO-style AP; run inference on unseen images; and export, reload and verify the adapter.

**This notebook does not demonstrate:** real-world traffic sign detection (the adaptation dataset is drawn in code, so the model learns these renderings and nothing about road photographs); COCO benchmark results (the average-precision helper here is a compact implementation without pycocotools area ranges or crowd handling, and it is run on synthetic data only); full-schedule YOLOS training (the upstream schedule is 200 epochs of ImageNet-1k pre-training and 150 epochs of COCO fine-tuning; the tutorial runs a few epochs on 30 images); segmentation; video tracking.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). A CUDA GPU such as a Colab or Kaggle T4 is the documented runtime for the fine-tuning stages and is used automatically when present; the notebook also runs on CPU, more slowly. Runtimes are not measured in this revision. The pinned `torch==2.14.0` wheel and the ~123 MB checkpoint are the largest downloads.
- **Knowledge:** basic Python and PIL; bounding boxes as xyxy pixel coordinates; intersection-over-union (IoU); and how to read average precision (AP50 and AP@[.50:.95]).
- **Data:** the default path generates everything in code with `samples.py` and downloads no dataset: one 640×480 COCO demonstration scene and a 40-image labelled sign dataset. BYOD is optional and off by default. Expected BYOD input: one image, or a directory holding `annotations.json` — a list of `{'file': 'name.png', 'boxes': [[x0, y0, x1, y1], ...], 'labels': [name, ...]}` objects — and the image files it names. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so; uploaded inputs stay in this runtime and are not sent to any inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `hustvl/yolos-small` snapshot (~123 MB in total) at revision `3d8f7130d3ce…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `scipy`, `numpy`, `PIL` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'scipy==1.18.1',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'yolos-detection-pipeline',
    'repository_revision': '128b25d4d0cd768156570e3655c4b4b51877fb67',
    'embedded_module': 'src/yolos_detection_pipeline/pipeline.py',
    'embedded_modules': ['src/yolos_detection_pipeline/pipeline.py', 'src/yolos_detection_pipeline/samples.py'],
    'module_sha256': '0285f08ede3d70f75dcd8f162b68972bd16632d9ffaf27762a532a31f6293e17',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, scipy, numpy, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'scipy': scipy.__version__, 'numpy': numpy.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/yolos_detection_pipeline/` @ `128b25d4d0cd`)

The next 2 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/2:** `src/yolos_detection_pipeline/pipeline.py`

In [ ]:
"""COCO object detection and bounded detection fine-tuning with the pinned YOLOS-Small checkpoint.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``),
always with ``trust_remote_code=False``: the YOLOS architecture (a plain ViT-S/16 with 100 detection
tokens) comes from the pinned ``transformers`` release, the weights are SafeTensors, and no
model-repository code is executed.

Until ``tools/pin_snapshot.py`` has recorded an immutable revision and every file's SHA-256, the package
refuses to stage, verify or load weights: an unpinned snapshot is never trusted.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "hustvl/yolos-small"
MODEL_REVISION = "3d8f7130d3ce4907cb206fe1c8485dc8fe8703de"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "yolos-small"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
ARTIFACT_FORMAT = "yolos-adapter-v1"
UNPINNED = "unpinned"
PIN_COMMAND = "python tools/pin_snapshot.py"

# The 91 category slots of the checkpoint's config.json id2label, in id order. The slots follow the
# original COCO category numbering; the eleven ids COCO 2017 never annotated (0, 12, 26, 29, 30, 45, 66,
# 68, 69, 71, 83) are all spelled "N/A" in this config, so the checkpoint was trained with boxes for the
# other 80 names only and "N/A" repeats.
LABELS = (
    "N/A",
    "person",
    "bicycle",
    "car",
    "motorcycle",
    "airplane",
    "bus",
    "train",
    "truck",
    "boat",
    "traffic light",
    "fire hydrant",
    "N/A",
    "stop sign",
    "parking meter",
    "bench",
    "bird",
    "cat",
    "dog",
    "horse",
    "sheep",
    "cow",
    "elephant",
    "bear",
    "zebra",
    "giraffe",
    "N/A",
    "backpack",
    "umbrella",
    "N/A",
    "N/A",
    "handbag",
    "tie",
    "suitcase",
    "frisbee",
    "skis",
    "snowboard",
    "sports ball",
    "kite",
    "baseball bat",
    "baseball glove",
    "skateboard",
    "surfboard",
    "tennis racket",
    "bottle",
    "N/A",
    "wine glass",
    "cup",
    "fork",
    "knife",
    "spoon",
    "bowl",
    "banana",
    "apple",
    "sandwich",
    "orange",
    "broccoli",
    "carrot",
    "hot dog",
    "pizza",
    "donut",
    "cake",
    "chair",
    "couch",
    "potted plant",
    "bed",
    "N/A",
    "dining table",
    "N/A",
    "N/A",
    "toilet",
    "N/A",
    "tv",
    "laptop",
    "mouse",
    "remote",
    "keyboard",
    "cell phone",
    "microwave",
    "oven",
    "toaster",
    "sink",
    "refrigerator",
    "N/A",
    "book",
    "clock",
    "vase",
    "scissors",
    "teddy bear",
    "hair drier",
    "toothbrush",
)
UNANNOTATED_LABEL_IDS: tuple[int, ...] = (0, 12, 26, 29, 30, 45, 66, 68, 69, 71, 83)

# Detection threshold: the value the Transformers YOLOS documentation example passes to
# post_process_object_detection (threshold=0.9); the pinned README shows no threshold. YOLOS scores each
# detection token with a softmax over its classes plus a "no object" class and reports the largest
# non-"no object" probability; the value was not calibrated for any deployment and the deployment owns
# tuning it on labelled images.
DETECTION_THRESHOLD = 0.9
EVAL_DETECTION_THRESHOLD = 0.05
MAX_DETECTIONS = 100
MAX_EVAL_DETECTIONS = 100
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MAX_CLASSES = 1000
MAX_RECORDS = 5000

# Training defaults for the bounded tutorial adaptation.
DEFAULT_EPOCHS = 10
DEFAULT_BATCH_SIZE = 4
DEFAULT_LEARNING_RATE = 1e-4
DEFAULT_WEIGHT_DECAY = 1e-4
DEFAULT_GRAD_CLIP = 0.1
DEFAULT_SEED = 20260924
# Bounded fine-tuning trains the last TRAINABLE_ENCODER_LAYERS encoder layers, the detection tokens, the
# position embeddings, the final layer norm and both heads; the patch embedding and the earlier layers
# keep their pretrained values.
TRAINABLE_ENCODER_LAYERS = 4
PATCH_EMBEDDING_PREFIX = "vit.embeddings.patch_embeddings."

COCO_IOU_THRESHOLDS: tuple[float, ...] = tuple(round(0.50 + 0.05 * i, 2) for i in range(10))


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def is_pinned() -> bool:
    """True once MODEL_REVISION names an immutable 40-hex commit."""
    revision = MODEL_REVISION
    return len(revision) == 40 and all(c in "0123456789abcdef" for c in revision)


def _require_pinned(action: str) -> None:
    if not is_pinned():
        raise RuntimeError(
            f"refusing to {action}: {MODEL_ID} has no pinned revision yet (MODEL_REVISION = "
            f"{MODEL_REVISION!r}); run `{PIN_COMMAND}` to record the commit and every file's SHA-256"
        )


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    _require_pinned("verify the snapshot")
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        if not entry.get("sha256"):
            raise ValueError(f"{entry['path']}: manifest records no sha256; run `{PIN_COMMAND}`")
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    _require_pinned("stage weights")
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def box_iou(a: Sequence[float], b: Sequence[float]) -> float:
    """Intersection-over-union of two xyxy pixel boxes; the building block for average precision."""
    if len(a) != 4 or len(b) != 4:
        raise ValueError("boxes must be [x0, y0, x1, y1]")
    if a[2] < a[0] or a[3] < a[1] or b[2] < b[0] or b[3] < b[1]:
        raise ValueError("boxes must satisfy x0 <= x1 and y0 <= y1")
    inter_w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return float(inter / union) if union > 0 else 0.0


def average_precision(
    predictions: Sequence[Sequence[Mapping[str, Any]]],
    references: Sequence[Mapping[str, Any]],
    class_names: Sequence[str],
    *,
    iou_thresholds: Sequence[float] = COCO_IOU_THRESHOLDS,
) -> dict[str, Any]:
    """Compact COCO-style average precision over a scored dataset.

    For each class and IoU threshold, detections are matched greedily to references by descending
    score; each detection matches at most one reference box, and precision is sampled at 101 recall
    points. Classes with no reference box are left out of the mean. There are no area ranges and no
    crowd handling, so the values are close to, but not identical with, pycocotools output.
    """
    if len(predictions) != len(references):
        raise ValueError(f"{len(predictions)} prediction lists but {len(references)} references")
    names = list(class_names)
    recall_points = np.linspace(0.0, 1.0, 101)
    per_threshold: dict[float, dict[str, float]] = {}

    for threshold in iou_thresholds:
        per_class: dict[str, float] = {}
        for name in names:
            scored: list[tuple[float, bool]] = []
            n_references = 0
            for dets, reference in zip(predictions, references, strict=True):
                ref_boxes = [
                    box
                    for box, label in zip(reference["boxes"], reference["labels"], strict=True)
                    if label == name
                ]
                n_references += len(ref_boxes)
                claimed = [False] * len(ref_boxes)
                candidates = sorted((d for d in dets if d["label"] == name), key=lambda d: -float(d["score"]))
                for det in candidates:
                    best, best_iou = -1, 0.0
                    for j, ref_box in enumerate(ref_boxes):
                        if claimed[j]:
                            continue
                        value = box_iou(det["box"], ref_box)
                        if value > best_iou:
                            best, best_iou = j, value
                    hit = best >= 0 and best_iou >= threshold
                    if hit:
                        claimed[best] = True
                    scored.append((float(det["score"]), hit))
            if n_references == 0:
                continue
            if not scored:
                per_class[name] = 0.0
                continue
            scored.sort(key=lambda pair: -pair[0])
            true_positives = np.cumsum([1 if hit else 0 for _score, hit in scored])
            false_positives = np.cumsum([0 if hit else 1 for _score, hit in scored])
            recall = true_positives / n_references
            precision = true_positives / np.maximum(true_positives + false_positives, 1)
            precision = np.maximum.accumulate(precision[::-1])[::-1]
            sampled = np.zeros_like(recall_points)
            indices = np.searchsorted(recall, recall_points, side="left")
            valid = indices < len(precision)
            sampled[valid] = precision[indices[valid]]
            per_class[name] = float(sampled.mean())
        per_threshold[threshold] = per_class

    scored_classes = sorted({name for values in per_threshold.values() for name in values})
    means = {
        threshold: (float(np.mean(list(values.values()))) if values else 0.0)
        for threshold, values in per_threshold.items()
    }
    return {
        "ap": float(np.mean(list(means.values()))) if means else 0.0,
        "ap50": means.get(0.5, 0.0),
        "ap75": means.get(0.75, 0.0),
        "per_class_ap50": per_threshold.get(0.5, {}),
        "iou_thresholds": [float(t) for t in iou_thresholds],
        "scored_classes": scored_classes,
        "n_images": len(references),
        "n_references": sum(len(r["boxes"]) for r in references),
    }


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(value: Any, name: str = "threshold") -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be a number in [0, 1], got {value!r}")
    return float(value)


def _check_class_names(class_names: Sequence[str]) -> tuple[str, ...]:
    names = tuple(class_names)
    if not 1 <= len(names) <= MAX_CLASSES:
        raise ValueError(f"class_names must hold 1..{MAX_CLASSES} names, got {len(names)}")
    for name in names:
        if not isinstance(name, str) or not name.strip():
            raise ValueError(f"class names must be non-empty strings, got {name!r}")
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        raise ValueError(f"class_names contains duplicates: {duplicates}")
    return names


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one image as PIL.Image.Image (any mode, converted to RGB): a photograph or a rendered scene",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "threshold": [0.0, 1.0],
    "labels": list(LABELS),
    "unannotated_label_ids": list(UNANNOTATED_LABEL_IDS),
    "max_detections": MAX_DETECTIONS,
    "preprocessing": (
        "image converted to RGB; the processor resizes so the shorter side is 800 px and the longer side at "
        "most 1333 px (aspect ratio preserved) and normalises with the ImageNet mean and standard deviation; "
        "returned boxes are mapped back to input pixels"
    ),
}

DATASET_SCHEMA: dict[str, Any] = {
    "record": "{'image': PIL.Image.Image, 'boxes': [[x0, y0, x1, y1], ...], 'labels': [class_name, ...]}",
    "boxes": "xyxy pixel coordinates inside the image, x0 < x1 and y0 < y1",
    "labels": "one name per box, each one of class_names",
    "records": [1, MAX_RECORDS],
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
}


def _check_inputs(image: Any, threshold: Any) -> tuple[Image.Image, float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request."""
    return validate_image(image), _check_threshold(threshold)


def validate_inputs(
    image: Image.Image,
    *,
    threshold: float = DETECTION_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict)."""
    _rgb, checked = _check_inputs(image, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (detect takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "threshold": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    class_names: Sequence[str],
    *,
    epochs: int = DEFAULT_EPOCHS,
) -> dict[str, Any]:
    """Validation stage for labelled detection records: raise on the first broken record, else return
    the dataset manifest. Classes that never occur are reported as findings, not silently accepted."""
    names = _check_class_names(class_names)
    if not records:
        raise ValueError("dataset must hold at least one record")
    if len(records) > MAX_RECORDS:
        raise ValueError(f"dataset holds {len(records)} records > MAX_RECORDS {MAX_RECORDS}")
    if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 100:
        raise ValueError(f"epochs must be an int in 1..100, got {epochs!r}")
    valid_classes = set(names)
    total_boxes = 0
    per_class = dict.fromkeys(names, 0)

    for idx, record in enumerate(records):
        if not isinstance(record, Mapping) or not {"image", "boxes", "labels"} <= set(record):
            raise ValueError(f"record {idx} must be a mapping with 'image', 'boxes' and 'labels'")
        try:
            img = validate_image(record["image"])
        except (TypeError, ValueError) as exc:
            raise type(exc)(f"record {idx}: {exc}") from exc
        boxes = record["boxes"]
        labels = record["labels"]
        if len(boxes) != len(labels):
            raise ValueError(f"record {idx}: {len(boxes)} boxes but {len(labels)} labels")
        width, height = img.size
        for b_idx, box in enumerate(boxes):
            if len(box) != 4:
                raise ValueError(f"record {idx} box {b_idx} must have 4 elements, got {len(box)}")
            x0, y0, x1, y1 = (float(v) for v in box)
            if not all(np.isfinite([x0, y0, x1, y1])):
                raise ValueError(f"record {idx} box {b_idx} has a non-finite coordinate")
            if not (0.0 <= x0 < x1 <= width and 0.0 <= y0 < y1 <= height):
                raise ValueError(
                    f"record {idx} box {b_idx} [{x0}, {y0}, {x1}, {y1}] is empty or "
                    f"outside image bounds {(width, height)}"
                )
        for label in labels:
            if label not in valid_classes:
                raise ValueError(f"record {idx} has unknown class {label!r}; expected one of {list(names)}")
            per_class[label] += 1
        total_boxes += len(boxes)

    absent = [name for name, count in per_class.items() if count == 0]
    findings = [f"class {name!r} has no box in this dataset" for name in absent]
    return {
        "schema": dict(DATASET_SCHEMA),
        "n_records": len(records),
        "n_boxes": total_boxes,
        "class_names": list(names),
        "boxes_per_class": per_class,
        "epochs": epochs,
        "findings": findings,
        "verdict": "accepted",
    }


def read_detection_records(directory: str | Path) -> list[dict[str, Any]]:
    """Read BYOD records from ``<directory>/annotations.json`` plus the image files it names.

    ``annotations.json`` is a list of ``{"file": "relative/name.png", "boxes": [[x0, y0, x1, y1], ...],
    "labels": [name, ...]}`` objects. File names must stay inside ``directory``; absolute paths and
    ``..`` segments are refused before any image is opened. The result still has to pass
    ``validate_dataset``.
    """
    root = Path(directory).resolve()
    index_path = root / "annotations.json"
    if not index_path.is_file():
        raise FileNotFoundError(f"{index_path} not found; expected annotations.json next to the images")
    with open(index_path, encoding="utf-8") as fh:
        entries = json.load(fh)
    if not isinstance(entries, list) or not entries:
        raise ValueError("annotations.json must hold a non-empty list of records")
    if len(entries) > MAX_RECORDS:
        raise ValueError(f"annotations.json lists {len(entries)} records > MAX_RECORDS {MAX_RECORDS}")
    records: list[dict[str, Any]] = []
    for idx, entry in enumerate(entries):
        if not isinstance(entry, dict) or not {"file", "boxes", "labels"} <= set(entry):
            raise ValueError(f"annotations.json entry {idx} must have 'file', 'boxes' and 'labels'")
        relative = Path(str(entry["file"]))
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(
                f"annotations.json entry {idx}: file {entry['file']!r} must be relative to {root}"
            )
        image_path = (root / relative).resolve()
        if root not in image_path.parents:
            raise ValueError(f"annotations.json entry {idx}: file {entry['file']!r} resolves outside {root}")
        with Image.open(image_path) as handle:
            image = handle.convert("RGB")
        records.append(
            {"image": image, "boxes": entry["boxes"], "labels": entry["labels"], "id": str(relative)}
        )
    return records


def evaluation_report(
    result: Mapping[str, Any],
    ground_truth_boxes: Mapping[str, Sequence[Sequence[float]]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Single-image evaluation stage: machine-readable report with per-object box_iou."""
    detections = list(result["detections"])
    labels = tuple(result.get("class_names") or LABELS)
    base = {
        "task": f"object detection over {len(labels)} class slots on one image",
        "decision_rule": (
            "a query survives when its largest class probability, taken from a softmax over the classes and "
            "a 'no object' class, reaches the threshold; the value is not a calibrated probability for the "
            "deployment's images, and each query reports exactly one label"
        ),
        "threshold": result.get("threshold", DETECTION_THRESHOLD),
        "sample_kind": sample_kind,
        "n_detections": len(detections),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not ground_truth_boxes:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth object boxes were supplied for the evaluated image",
            "needs": (
                "labelled boxes per class on your own images, scored per object with box_iou and aggregated "
                "into COCO-style average precision (AP@[.50:.95], AP50) at stated IoU thresholds"
            ),
        }
    metrics = []
    for label, boxes in ground_truth_boxes.items():
        if label not in labels:
            raise ValueError(f"unknown reference label {label!r}; expected one of {len(labels)} classes")
        same_label = [det for det in detections if det["label"] == label]
        for index, box in enumerate(boxes):
            ious = [box_iou(det["box"], box) for det in same_label]
            best = max(range(len(ious)), key=ious.__getitem__) if ious else None
            metrics.append(
                {
                    "id": "box_iou",
                    "reference": f"{label}-{index}",
                    "value": ious[best] if best is not None else 0.0,
                    "matched_score": same_label[best]["score"] if best is not None else None,
                    "n_detected_same_label": len(same_label),
                    "estimation": "one reference box per object on a single image, no dispersion estimate",
                }
            )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} reference box(es) on one tutorial image; geometry sanity evidence, "
            "not a detection benchmark"
        ),
        "needs": (
            "a labelled image set from the deployment domain (cameras, scenes, object classes) for any "
            "average-precision or precision/recall claim"
        ),
    }


def frozen_prefixes_for(num_hidden_layers: int) -> tuple[str, ...]:
    """Parameter-name prefixes the bounded fine-tune keeps frozen, for an encoder of this depth."""
    n_frozen = max(0, num_hidden_layers - TRAINABLE_ENCODER_LAYERS)
    return (PATCH_EMBEDDING_PREFIX, *(f"vit.encoder.layer.{i}." for i in range(n_frozen)))


def _coco_annotation(index: int, record: Mapping[str, Any], class_to_id: Mapping[str, int]) -> dict[str, Any]:
    """One record as the COCO-detection annotation the YOLOS processor converts into training targets."""
    annotations = []
    for box, label in zip(record["boxes"], record["labels"], strict=True):
        x0, y0, x1, y1 = (float(v) for v in box)
        annotations.append(
            {
                "bbox": [x0, y0, x1 - x0, y1 - y0],
                "category_id": class_to_id[label],
                "area": (x1 - x0) * (y1 - y0),
                "iscrowd": 0,
            }
        )
    return {"image_id": index, "annotations": annotations}


@dataclass
class YolosDetectionPipeline:
    """COCO-class object detection and transfer fine-tuning over YOLOS-Small."""

    model: Any
    processor: Any
    device: str
    class_names: tuple[str, ...] = LABELS
    source: str = "snapshot"
    base_state_digest: str | None = None
    adapted: bool = False
    reinitialised: tuple[str, ...] = field(default_factory=tuple)
    frozen_prefixes: tuple[str, ...] = field(default_factory=tuple)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        class_names: Sequence[str] | None = None,
        seed: int = DEFAULT_SEED,
    ) -> YolosDetectionPipeline:
        _require_pinned("load the model")
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(
                f"no snapshot manifest at {root}; stage {MODEL_ID}@{MODEL_REVISION} "
                f"under weights/{MODEL_KEY} "
                "(allow_download=True fetches the manifest-listed files)"
            )
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        names = _check_class_names(class_names) if class_names is not None else LABELS

        import torch
        from transformers import YolosForObjectDetection, YolosImageProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        common = {"trust_remote_code": False, "local_files_only": True}
        processor = YolosImageProcessor.from_pretrained(str(root), **common)
        base_digest = _sha256(root / "model.safetensors") if (root / "model.safetensors").is_file() else None

        reinitialised: tuple[str, ...] = ()
        if class_names is not None:
            # Deterministic re-heading: the class head is a 3-layer MLP, and only its last layer changes
            # shape (num_labels + 1 outputs, the last being "no object"), so only that layer is freshly
            # initialised; every other tensor, including the MLP's first two layers, comes from the snapshot.
            torch.manual_seed(seed)
            model = YolosForObjectDetection.from_pretrained(
                str(root),
                num_labels=len(names),
                id2label=dict(enumerate(names)),
                label2id={name: i for i, name in enumerate(names)},
                ignore_mismatched_sizes=True,
                **common,
            )
            reinitialised = (
                "class_labels_classifier.layers.2.weight",
                "class_labels_classifier.layers.2.bias",
            )
        else:
            model = YolosForObjectDetection.from_pretrained(str(root), **common)

        model = model.to(resolved_device).eval()
        return cls(
            model=model,
            processor=processor,
            device=resolved_device,
            class_names=names,
            source=str(root),
            base_state_digest=base_digest,
            adapted=False,
            reinitialised=reinitialised,
        )

    def _run(self, image: Image.Image, threshold: float) -> list[dict[str, Any]]:
        import torch

        inputs = self.processor(images=image, return_tensors="pt")
        was_training = self.model.training
        self.model.eval()
        with torch.inference_mode():
            outputs = self.model(pixel_values=inputs["pixel_values"].to(self.device))
        if was_training:
            self.model.train()
        result = self.processor.post_process_object_detection(
            outputs, threshold=threshold, target_sizes=[(image.height, image.width)]
        )[0]
        detections = []
        for box, label_idx, score in zip(result["boxes"], result["labels"], result["scores"], strict=True):
            idx = int(label_idx)
            label_name = self.class_names[idx] if idx < len(self.class_names) else f"class_{idx}"
            detections.append(
                {"box": [float(v) for v in box.tolist()], "label": label_name, "score": float(score)}
            )
        return detections

    def detect(self, image: Image.Image, *, threshold: float = DETECTION_THRESHOLD) -> dict[str, Any]:
        """Detect objects on one image; boxes are xyxy pixel coordinates in the input image."""
        rgb, checked = _check_inputs(image, threshold)
        detections = self._run(rgb, checked)
        if len(detections) > MAX_DETECTIONS:
            raise RuntimeError(
                f"backend returned {len(detections)} detections > num_detection_tokens {MAX_DETECTIONS}"
            )
        for det in detections:
            if (
                set(det) != {"box", "label", "score"}
                or len(det["box"]) != 4
                or det["label"] not in self.class_names
            ):
                raise RuntimeError(f"backend returned a malformed detection: {det!r}")
        return {
            "detections": sorted(detections, key=lambda d: -d["score"]),
            "threshold": checked,
            "width": rgb.width,
            "height": rgb.height,
            "class_names": list(self.class_names),
            "adapted": self.adapted,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def detect_many(
        self,
        images: Sequence[Image.Image],
        *,
        threshold: float = EVAL_DETECTION_THRESHOLD,
    ) -> list[list[dict[str, Any]]]:
        """Detections for several images, defaulting to the evaluation threshold."""
        return [self.detect(image, threshold=threshold)["detections"] for image in images]

    def finetune(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        epochs: int = DEFAULT_EPOCHS,
        batch_size: int = DEFAULT_BATCH_SIZE,
        learning_rate: float = DEFAULT_LEARNING_RATE,
        seed: int = DEFAULT_SEED,
        freeze_early_layers: bool = True,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning on ``records`` with the DETR set-prediction loss YOLOS was trained with.

        The loss is the upstream ``YolosForObjectDetection`` loss: Hungarian matching of detection tokens to
        reference boxes, then cross-entropy over the classes plus "no object" (weighted by the config's
        ``eos_coefficient``), L1 and generalised-IoU box terms. With ``freeze_early_layers`` the patch
        embedding and all but the last ``TRAINABLE_ENCODER_LAYERS`` encoder layers stay frozen. Mutates this
        pipeline in place (``adapted`` becomes True) and leaves the model in eval mode.
        """
        import torch

        validate_dataset(records, self.class_names, epochs=epochs)
        if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size < 1:
            raise ValueError(f"batch_size must be a positive int, got {batch_size!r}")
        if isinstance(learning_rate, bool) or not isinstance(learning_rate, int | float):
            raise ValueError(f"learning_rate must be a number, got {learning_rate!r}")
        if not 0.0 < float(learning_rate) <= 1.0:
            raise ValueError(f"learning_rate must be in (0, 1], got {learning_rate!r}")

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        rng = np.random.default_rng(seed)

        prefixes = frozen_prefixes_for(self.model.config.num_hidden_layers) if freeze_early_layers else ()
        for name, parameter in self.model.named_parameters():
            parameter.requires_grad = not (prefixes and name.startswith(prefixes))
        self.frozen_prefixes = prefixes
        trainable = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable, lr=float(learning_rate), weight_decay=DEFAULT_WEIGHT_DECAY)
        class_to_id = {name: i for i, name in enumerate(self.class_names)}

        epoch_losses: list[float] = []
        for epoch in range(epochs):
            self.model.train()
            order = rng.permutation(len(records))
            running_loss, n_batches = 0.0, 0
            for start in range(0, len(records), batch_size):
                batch = [records[int(i)] for i in order[start : start + batch_size]]
                images = [validate_image(r["image"]) for r in batch]
                annotations = [_coco_annotation(start + k, r, class_to_id) for k, r in enumerate(batch)]
                encoded = self.processor(images=images, annotations=annotations, return_tensors="pt")
                labels = [
                    {key: value.to(self.device) for key, value in target.items()}
                    for target in encoded["labels"]
                ]
                optimizer.zero_grad(set_to_none=True)
                outputs = self.model(pixel_values=encoded["pixel_values"].to(self.device), labels=labels)
                loss = outputs.loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(trainable, DEFAULT_GRAD_CLIP)
                optimizer.step()
                running_loss += float(loss.detach().cpu())
                n_batches += 1
            epoch_losses.append(running_loss / max(1, n_batches))
            if progress is not None:
                progress({"epoch": epoch + 1, "epochs": epochs, "loss": epoch_losses[-1]})

        self.model.eval()
        self.adapted = True
        return {
            "epochs": epochs,
            "batch_size": batch_size,
            "learning_rate": float(learning_rate),
            "weight_decay": DEFAULT_WEIGHT_DECAY,
            "grad_clip_norm": DEFAULT_GRAD_CLIP,
            "optimizer": "AdamW",
            "seed": seed,
            "precision": "float32",
            "freeze_early_layers": freeze_early_layers,
            "trainable_encoder_layers": (
                min(TRAINABLE_ENCODER_LAYERS, self.model.config.num_hidden_layers)
                if freeze_early_layers
                else self.model.config.num_hidden_layers
            ),
            "frozen_prefixes": list(self.frozen_prefixes),
            "trainable_parameters": sum(p.numel() for p in trainable),
            "total_parameters": sum(p.numel() for p in self.model.parameters()),
            "epoch_losses": epoch_losses,
            "final_loss": epoch_losses[-1] if epoch_losses else None,
            "loss": "upstream YolosForObjectDetection loss (Hungarian matching; cross-entropy + L1 + GIoU)",
            "device": self.device,
            "class_names": list(self.class_names),
            "reinitialised_tensors": list(self.reinitialised),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        threshold: float = EVAL_DETECTION_THRESHOLD,
        iou_thresholds: Sequence[float] = COCO_IOU_THRESHOLDS,
        max_detections: int = MAX_EVAL_DETECTIONS,
    ) -> dict[str, Any]:
        """Score a labelled dataset: average precision plus evaluation metadata."""
        checked = _check_threshold(threshold, "threshold")
        predictions = self.detect_many([r["image"] for r in records], threshold=checked)
        raw_counts = [len(dets) for dets in predictions]
        capped = [dets[:max_detections] for dets in predictions]
        metrics = average_precision(capped, records, self.class_names, iou_thresholds=iou_thresholds)
        return {
            **metrics,
            "threshold": checked,
            "max_detections": max_detections,
            "detections_before_cap": raw_counts,
            "adapted": self.adapted,
            "class_names": list(self.class_names),
            "estimation": (
                f"one pass over {len(records)} held-out images; no resampling, no dispersion estimate"
            ),
            "implementation": (
                "package-local average_precision: greedy score-ordered matching and 101-point interpolation, "
                "without pycocotools area ranges or crowd handling"
            ),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def save_artifact(self, path: str | Path, *, notes: str | None = None) -> dict[str, Any]:
        """Write the adapted tensors as one SafeTensors file with the provenance in its metadata.

        Tensors under ``frozen_prefixes`` are left out: they equal the verified base snapshot, which
        ``load_artifact`` loads first. The artifact is therefore an adapter bound to the base revision.
        """
        from safetensors.torch import save_file

        if not self.adapted:
            raise RuntimeError("nothing to export: the pipeline has not been fine-tuned")
        artifact_path = Path(path)
        artifact_path.parent.mkdir(parents=True, exist_ok=True)
        tensors = {
            name: value.detach().cpu().contiguous().clone()
            for name, value in self.model.state_dict().items()
            if not any(name.startswith(prefix) for prefix in self.frozen_prefixes)
        }
        metadata = {
            "format": ARTIFACT_FORMAT,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "model_key": MODEL_KEY,
            "class_names": json.dumps(list(self.class_names)),
            "frozen_prefixes": json.dumps(list(self.frozen_prefixes)),
            "base_state_digest": self.base_state_digest or "",
            "notes": notes or "",
        }
        save_file(tensors, str(artifact_path), metadata=metadata)
        return {
            "path": str(artifact_path),
            "bytes": artifact_path.stat().st_size,
            "sha256": _sha256(artifact_path),
            "format": ARTIFACT_FORMAT,
            "class_names": list(self.class_names),
            "tensors": len(tensors),
            "frozen_prefixes": list(self.frozen_prefixes),
            "base_state_digest": self.base_state_digest,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    @staticmethod
    def read_artifact_metadata(path: str | Path) -> dict[str, Any]:
        """Read and check the artifact's provenance header without loading any tensor."""
        from safetensors import safe_open

        with safe_open(str(path), framework="pt") as handle:
            metadata = dict(handle.metadata() or {})
        if metadata.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {metadata.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if (metadata.get("model_id"), metadata.get("model_revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError(
                f"artifact was built on {metadata.get('model_id')}@{metadata.get('model_revision')}, "
                f"package pins {MODEL_ID}@{MODEL_REVISION}"
            )
        if metadata.get("model_key") != MODEL_KEY:
            raise ValueError(
                f"artifact was built on {metadata.get('model_key')!r}, package pins {MODEL_KEY!r}"
            )
        return {
            **metadata,
            "class_names": json.loads(metadata["class_names"]),
            "frozen_prefixes": json.loads(metadata["frozen_prefixes"]),
        }

    def apply_artifact(self, path: str | Path) -> None:
        """Load adapter tensors onto this (base) pipeline; refuse any tensor set that does not fit."""
        from safetensors.torch import load_file

        metadata = self.read_artifact_metadata(path)
        if tuple(metadata["class_names"]) != tuple(self.class_names):
            raise ValueError("artifact class_names differ from this pipeline's class_names")
        expected_base = metadata.get("base_state_digest") or None
        if expected_base and self.base_state_digest and expected_base != self.base_state_digest:
            raise ValueError("artifact was exported against a different base model.safetensors digest")
        tensors = load_file(str(path), device="cpu")
        prefixes = tuple(metadata["frozen_prefixes"])
        result = self.model.load_state_dict(tensors, strict=False)
        if result.unexpected_keys:
            raise ValueError(
                f"artifact carries tensors the model does not have: {result.unexpected_keys[:5]}"
            )
        stray = [key for key in result.missing_keys if not any(key.startswith(p) for p in prefixes)]
        if stray:
            raise ValueError(f"artifact is missing trainable tensors: {stray[:5]}")
        self.model.eval()
        self.adapted = True
        self.frozen_prefixes = prefixes
        self.source = f"artifact:{Path(path).name}"

    @classmethod
    def load_artifact(
        cls,
        path: str | Path,
        *,
        weights_dir: str | Path | None = None,
        device: str | None = None,
    ) -> YolosDetectionPipeline:
        """Rebuild an adapted pipeline: verified base snapshot first, then the adapter tensors."""
        metadata = cls.read_artifact_metadata(path)
        pipe = cls.from_pretrained(
            device=device, weights_dir=weights_dir, class_names=metadata["class_names"]
        )
        pipe.apply_artifact(path)
        return pipe

**Module 2/2:** `src/yolos_detection_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Deterministic in-code sample data: the COCO demonstration scene and the adaptation dataset.

Nothing here is downloaded and nothing needs torch — Pillow and numpy only — so the tutorial's
default path has no dataset dependency and the same generators are exercised by the repository's unit
tests.

Two separate label vocabularies live here and must not be conflated:

* ``tutorial_scene`` returns references drawn from the checkpoint's own **COCO** classes; it
  demonstrates the pretrained model and is not training data.
* ``sign_dataset`` returns records labelled with ``SIGN_CLASSES``, a three-class vocabulary that does
  **not** exist in COCO. It is the adaptation dataset, and a model fine-tuned on it answers in those
  three names only.
"""

from __future__ import annotations

import math
from typing import Any

import numpy as np
from PIL import Image, ImageDraw, ImageFont

COCO_SCENE_SIZE = (640, 480)
ADAPT_SCENE_SIZE = (640, 640)

# The adaptation vocabulary. Deliberately not COCO names: "stop sign" exists in COCO, "yield-sign" and
# "speed-limit-sign" do not, and the hyphenated spellings keep the two vocabularies visually distinct
# in output. A model fine-tuned on this dataset predicts only these three.
SIGN_CLASSES: tuple[str, ...] = ("stop-sign", "yield-sign", "speed-limit-sign")


def _font(size: int) -> Any:
    return ImageFont.load_default(size=size)


def tutorial_scene(
    width: int = COCO_SCENE_SIZE[0], height: int = COCO_SCENE_SIZE[1]
) -> tuple[Image.Image, dict[str, list[list[float]]]]:
    """The COCO demonstration scene: drawn objects and their reference boxes, keyed by COCO label.

    These are drawn references on a rendered picture, not a labelled photographic dataset: they are
    enough for a per-object ``box_iou`` sanity check and nothing more.
    """
    img = Image.new("RGB", (width, height), (135, 190, 235))
    d = ImageDraw.Draw(img)
    d.rectangle([0, int(height * 0.6875), width, height], fill=(96, 128, 72))
    d.rectangle([0, int(height * 0.625), width, int(height * 0.6875)], fill=(110, 110, 110))
    refs: dict[str, list[list[float]]] = {}

    # Stop sign
    cx, cy, r = 110, 150, 62
    pts = [
        (cx + r * math.cos(math.pi / 8 + k * math.pi / 4), cy + r * math.sin(math.pi / 8 + k * math.pi / 4))
        for k in range(8)
    ]
    d.rectangle([cx - 5, cy, cx + 5, int(height * 0.6875)], fill=(90, 90, 90))
    d.polygon(pts, fill=(200, 20, 30), outline=(255, 255, 255))
    f = _font(30)
    d.text((cx - d.textlength("STOP", font=f) / 2, cy - 17), "STOP", fill="white", font=f)
    refs["stop sign"] = [[float(cx - r), float(cy - r), float(cx + r), float(cy + r)]]

    # Traffic light
    x0, y0 = 270, 60
    d.rectangle([x0 + 22, y0 + 150, x0 + 30, int(height * 0.6875)], fill=(70, 70, 70))
    d.rectangle([x0, y0, x0 + 52, y0 + 150], fill=(25, 25, 25), outline=(60, 60, 60))
    for k, col in enumerate([(230, 30, 30), (240, 200, 30), (40, 200, 60)]):
        d.ellipse([x0 + 8, y0 + 8 + k * 47, x0 + 44, y0 + 44 + k * 47], fill=col)
    refs["traffic light"] = [[float(x0), float(y0), float(x0 + 52), float(y0 + 150)]]

    # Clock
    cx, cy, r = 480, 140, 70
    d.rectangle([cx - 6, cy, cx + 6, int(height * 0.6875)], fill=(120, 80, 40))
    d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(250, 250, 245), outline=(20, 20, 20), width=5)
    f2 = _font(16)
    for h in range(1, 13):
        a = math.radians(h * 30 - 90)
        d.text(
            (cx + (r - 18) * math.cos(a) - 5, cy + (r - 18) * math.sin(a) - 8),
            str(h),
            fill="black",
            font=f2,
        )
    d.line(
        [(cx, cy), (cx + 0.5 * r * math.cos(math.radians(-60)), cy + 0.5 * r * math.sin(math.radians(-60)))],
        fill="black",
        width=5,
    )
    d.line(
        [(cx, cy), (cx + 0.8 * r * math.cos(math.radians(30)), cy + 0.8 * r * math.sin(math.radians(30)))],
        fill="black",
        width=3,
    )
    refs["clock"] = [[float(cx - r), float(cy - r), float(cx + r), float(cy + r)]]

    # Sports ball
    cx, cy, r = 330, 400, 45
    d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(235, 120, 30), outline=(40, 20, 10), width=3)
    d.line([(cx - r, cy), (cx + r, cy)], fill=(40, 20, 10), width=3)
    d.line([(cx, cy - r), (cx, cy + r)], fill=(40, 20, 10), width=3)
    d.arc([cx - r * 1.6, cy - r, cx - r * 0.2, cy + r], 300, 60, fill=(40, 20, 10), width=3)
    d.arc([cx + r * 0.2, cy - r, cx + r * 1.6, cy + r], 120, 240, fill=(40, 20, 10), width=3)
    refs["sports ball"] = [[float(cx - r), float(cy - r), float(cx + r), float(cy + r)]]

    return img, refs


def blank_scene(width: int = 640, height: int = 640) -> Image.Image:
    """A featureless white image: the degenerate input every detector should be asked about."""
    return Image.new("RGB", (width, height), (255, 255, 255))


def noise_scene(seed: int = 0, width: int = 640, height: int = 640) -> Image.Image:
    """Uniform RGB noise: structure-free input, for the same reason as ``blank_scene``."""
    rng = np.random.default_rng(seed)
    return Image.fromarray(rng.integers(0, 256, (height, width, 3), dtype=np.uint8))


def _octagon(draw: ImageDraw.ImageDraw, cx: float, cy: float, r: float, font: Any) -> list[float]:
    points = [
        (cx + r * math.cos(math.pi / 8 + i * math.pi / 4), cy + r * math.sin(math.pi / 8 + i * math.pi / 4))
        for i in range(8)
    ]
    draw.polygon(points, fill=(196, 30, 34), outline=(255, 255, 255))
    text = "STOP"
    draw.text(
        (cx - draw.textlength(text, font=font) / 2, cy - r * 0.27), text, fill=(255, 255, 255), font=font
    )
    half = r * math.cos(math.pi / 8)
    return [cx - half, cy - half, cx + half, cy + half]


def _yield_sign(draw: ImageDraw.ImageDraw, cx: float, cy: float, r: float) -> list[float]:
    points = [(cx, cy + r), (cx - r * 0.95, cy - r * 0.75), (cx + r * 0.95, cy - r * 0.75)]
    draw.polygon(points, fill=(255, 255, 255), outline=(198, 32, 36))
    inner = [(cx, cy + r * 0.62), (cx - r * 0.62, cy - r * 0.5), (cx + r * 0.62, cy - r * 0.5)]
    draw.line([*inner, inner[0]], fill=(198, 32, 36), width=int(max(4, r * 0.22)))
    return [cx - r * 0.95, cy - r * 0.75, cx + r * 0.95, cy + r]


def _speed_limit_sign(draw: ImageDraw.ImageDraw, cx: float, cy: float, r: float, limit: int) -> list[float]:
    draw.ellipse(
        [cx - r, cy - r, cx + r, cy + r],
        fill=(255, 255, 255),
        outline=(198, 32, 36),
        width=int(max(4, r * 0.2)),
    )
    font = _font(int(max(12, r * 0.9)))
    text = str(limit)
    draw.text((cx - draw.textlength(text, font=font) / 2, cy - r * 0.55), text, fill=(30, 30, 30), font=font)
    return [cx - r, cy - r, cx + r, cy + r]


def sign_dataset(
    n_images: int = 40,
    *,
    seed: int = 0,
    max_objects: int = 3,
    size: tuple[int, int] = ADAPT_SCENE_SIZE,
) -> list[dict[str, Any]]:
    """A deterministic labelled dataset over ``SIGN_CLASSES`` for the bounded fine-tune.

    Each record is ``{"id": str, "image": PIL.Image, "boxes": [[x0, y0, x1, y1], ...], "labels": [name,
    ...]}`` — the record shape ``validate_dataset`` and ``finetune`` accept, and the shape a BYOD caller
    must produce from their own labelled images. Signs are placed on a non-overlapping grid so the boxes
    are exact by construction rather than approximate.

    It is synthetic drawn data, so a model fine-tuned on it learns to find *these renderings*. That is
    the point of a bounded tutorial adaptation and the reason its metrics are not a claim about real
    traffic signs.
    """
    if not 1 <= n_images <= 500:
        raise ValueError(f"n_images must be in 1..500, got {n_images}")
    if not 1 <= max_objects <= 6:
        raise ValueError(f"max_objects must be in 1..6, got {max_objects}")
    rng = np.random.default_rng(seed)
    width, height = size
    slots = [(x, y) for y in (150, 400) for x in (140, 360, 560)]
    records: list[dict[str, Any]] = []
    for index in range(n_images):
        image = Image.new("RGB", size, (232, 236, 240))
        draw = ImageDraw.Draw(image)
        tint = rng.integers(200, 245, 3)
        draw.rectangle([0, 0, width, height], fill=tuple(int(v) for v in tint))
        draw.rectangle([0, int(height * 0.72), width, height], fill=(118, 122, 126))
        n_objects = int(rng.integers(1, max_objects + 1))
        chosen = rng.permutation(len(slots))[:n_objects]
        boxes: list[list[float]] = []
        labels: list[str] = []
        for slot_index in chosen:
            cx, cy = slots[int(slot_index)]
            cx += float(rng.integers(-28, 29))
            cy += float(rng.integers(-28, 29))
            radius = float(rng.integers(46, 71))
            cx = min(max(cx, radius + 6), width - radius - 6)
            cy = min(max(cy, radius + 6), height - radius - 100)
            kind = SIGN_CLASSES[int(rng.integers(0, len(SIGN_CLASSES)))]
            draw.rectangle(
                [cx - 5, cy + radius * 0.6, cx + 5, min(height - 1, cy + radius + 90)], fill=(112, 112, 116)
            )
            if kind == "stop-sign":
                box = _octagon(draw, cx, cy, radius, _font(int(max(11, radius * 0.34))))
            elif kind == "yield-sign":
                box = _yield_sign(draw, cx, cy, radius)
            else:
                box = _speed_limit_sign(draw, cx, cy, radius, int(rng.choice([30, 50, 60, 80])))
            boxes.append(
                [
                    float(max(0.0, box[0])),
                    float(max(0.0, box[1])),
                    float(min(width, box[2])),
                    float(min(height, box[3])),
                ]
            )
            labels.append(kind)
        records.append({"id": f"sign-{seed}-{index:03d}", "image": image, "boxes": boxes, "labels": labels})
    return records


def split_dataset(
    records: list[dict[str, Any]],
    *,
    train_fraction: float = 0.75,
    seed: int = 0,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Deterministically partition records into train and held-out splits (random, record-level).

    A random split assumes the records are independent. Records that share a source photograph, a
    scene or a camera session must be split by that group instead, or the held-out score leaks.
    """
    if not 0.0 < train_fraction < 1.0:
        raise ValueError(f"train_fraction must be between 0 and 1, got {train_fraction}")
    if len(records) < 2:
        raise ValueError(f"at least 2 records are required to split, got {len(records)}")
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(records))
    n_train = min(len(records) - 1, max(1, int(len(records) * train_fraction)))
    train_idx = {int(i) for i in indices[:n_train]}
    train = [r for i, r in enumerate(records) if i in train_idx]
    held_out = [r for i, r in enumerate(records) if i not in train_idx]
    return train, held_out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `3d8f7130d3ce…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `YolosDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "yolos-small",
  "modelId": "hustvl/yolos-small",
  "revision": "3d8f7130d3ce4907cb206fe1c8485dc8fe8703de",
  "files": [
    {
      "path": "README.md",
      "bytes": 4173,
      "sha256": "0e95447d56c12b6833a42bc781fefcb318c65e6499d1b7b03912cbbfbc1c6352"
    },
    {
      "path": "config.json",
      "bytes": 4132,
      "sha256": "340c26d18da821b80b7e3a558697f0d39328f99f9f4eabe246ea9851e7f50de5"
    },
    {
      "path": "model.safetensors",
      "bytes": 122763274,
      "sha256": "27823dd210017c83fb2abb5528b7344d8b73572b5764d20f9cda2c153727232c"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 292,
      "sha256": "27645e9dec37d884357ee5bf9eab8b5464c71cbb6660a56497e91caca681fdd4"
    }
  ],
  "totalBytes": 122771871
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = YolosDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. What the pretrained detector does on a drawn scene

Before adapting anything, inspect the model you start from. The carried `samples` module draws a deterministic street scene with four objects and their reference boxes: a **stop sign**, a **traffic light**, an analogue **clock**, and an orange **sports ball**. All four are COCO classes.

The detection threshold is a **caller-owned request parameter**, not a pipeline constant. Each score is the query's largest class probability from a **softmax over the classes and a no-object class, not a calibrated** probability for your images. The default `0.9` is the value the pinned model README uses, and it is passed explicitly on every call.

The scene is drawn, not photographed, so a miss here is a finding about renderings, not about photographs. The evaluation report records every miss with `box_iou = 0.0`. COCO mean average precision needs a labelled image set; on one scene the verdict is `sample-sanity`.

In [ ]:
import hashlib
import io
import json
import os
from pathlib import Path

os.makedirs('outputs', exist_ok=True)
OUTPUTS = Path('outputs')

threshold = 0.9  # @param {type:"number"}

print({'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_DETECTIONS': MAX_DETECTIONS,
       'label_slots': len(LABELS), 'unannotated_slots': len(UNANNOTATED_LABEL_IDS), 'DETECTION_THRESHOLD': DETECTION_THRESHOLD})

scene, references = tutorial_scene()
buffer = io.BytesIO()
scene.save(buffer, format='PNG')
print({'sample_kind': 'synthetic', 'size': list(scene.size), 'sha256': hashlib.sha256(buffer.getvalue()).hexdigest()[:16],
       'references': {label: len(boxes) for label, boxes in references.items()}})

input_manifest = validate_inputs(scene, threshold=threshold, names=['tutorial-scene'])
try:
    validate_inputs(scene, threshold=1.5)
except ValueError as exc:
    input_manifest['findings'].append({'probe': 'threshold=1.5', 'rejected': str(exc)})
print({'verdict': input_manifest['verdict'], 'findings': input_manifest['findings'], 'inputs': input_manifest['inputs']})

coco_result = pipe.detect(scene, threshold=threshold)
for det in coco_result['detections']:
    print(f"{det['label']:>14s} {det['score']:.3f}  [{', '.join(f'{v:.0f}' for v in det['box'])}]")

coco_report = evaluation_report(coco_result, references, sample_kind='synthetic')
print({'verdict': coco_report['verdict'], 'n_detections': coco_report['n_detections']})
for metric in coco_report['metrics']:
    print(f"  {metric['reference']:>18s}  box_iou {metric['value']:.3f}  same-label detections {metric['n_detected_same_label']}")
hits = sum(1 for m in coco_report['metrics'] if m['value'] >= 0.5)
print(f'{hits}/{len(coco_report["metrics"])} drawn objects matched at IoU >= 0.5')
scene

## 5. Degenerate input probes: blank canvas and noise

Ask a detector about structure-free input before trusting it. A model that returns confident boxes on a blank canvas or on uniform noise will return them on empty real frames too. The cell counts detections on both images at the default threshold and at the evaluation threshold `EVAL_DETECTION_THRESHOLD` (0.05) that average precision is computed at.

**What to look for:** any detection above the default threshold on these two images is a false positive by construction.

In [ ]:
degenerate = {}
for name, image in (('blank', blank_scene()), ('noise', noise_scene(0))):
    standard = pipe.detect(image, threshold=threshold)['detections']
    lenient = pipe.detect(image, threshold=EVAL_DETECTION_THRESHOLD)['detections']
    degenerate[name] = {
        'at_default_threshold': len(standard),
        'at_evaluation_threshold': len(lenient),
        'top': [(d['label'], round(d['score'], 3)) for d in lenient[:3]],
    }
print(json.dumps(degenerate, indent=2))

## 6. Labelled adaptation dataset and validation

Suppose your task needs sign classes that COCO does not have. `sign_dataset` draws a deterministic 40-image dataset over `SIGN_CLASSES`: `stop-sign`, `yield-sign` and `speed-limit-sign`.

**Keep the two vocabularies apart.** COCO has `stop sign` (with a space). The adaptation vocabulary uses `stop-sign` (hyphenated), `yield-sign` and `speed-limit-sign`. They are different class identities, and the adapted model answers in the new names only.

`validate_dataset` checks every record before any model runs: the record keys, the image size ceilings, that every box is finite, non-empty and inside its image, and that every label is in the vocabulary. It returns a dataset manifest with the box count per class and a finding for any class that has no box.

In [ ]:
N_IMAGES = 40  # @param {type:"integer"}
DATASET_SEED = 0  # @param {type:"integer"}
EPOCHS = 10  # @param {type:"integer"}

records = sign_dataset(N_IMAGES, seed=DATASET_SEED)
dataset_manifest = validate_dataset(records, SIGN_CLASSES, epochs=EPOCHS)
print(json.dumps({k: v for k, v in dataset_manifest.items() if k != 'schema'}, indent=2))

preview = Image.new('RGB', (480, 320))
for index, record in enumerate(records[:6]):
    preview.paste(record['image'].resize((160, 160)), (160 * (index % 3), 160 * (index // 3)))
preview

## 7. Split, re-head, and measure the pre-adaptation baseline

The dataset is split at random into a training part (75%, 30 images) and a held-out part (25%, 10 images). A random split is valid here because every image is drawn independently; records that share a photograph or a camera session must be split by that group instead. The held-out images are never shown to the optimizer, and no hyperparameter is selected on them.

`from_pretrained(class_names=SIGN_CLASSES)` loads the verified checkpoint and replaces only the last layer of the three-layer class head (`class_labels_classifier.layers.2`, now 3 classes plus no-object) with a freshly initialised layer. The encoder, the detection tokens, the first two class-head layers and the box head keep their COCO weights.

**The baseline is expected to be near zero.** A randomly initialised class head has no information about the new classes, so this number is the floor the fine-tune has to beat, not a property of YOLOS.

In [ ]:
HOLDOUT = 0.25  # @param {type:"number"}
SEED = 0  # @param {type:"integer"}

train_records, held_out = split_dataset(records, train_fraction=1.0 - HOLDOUT, seed=SEED)
overlap = {r['id'] for r in train_records} & {r['id'] for r in held_out}
assert not overlap, f'split leaked records: {sorted(overlap)}'
print({'train': len(train_records), 'held_out': len(held_out),
       'train_boxes': sum(len(r['boxes']) for r in train_records),
       'held_out_boxes': sum(len(r['boxes']) for r in held_out)})

adapter = YolosDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, class_names=SIGN_CLASSES, seed=SEED)
print({'class_names': list(adapter.class_names), 'device': adapter.device, 'reinitialised': list(adapter.reinitialised)})

baseline = adapter.evaluate(held_out)
print(json.dumps({'ap': round(baseline['ap'], 4), 'ap50': round(baseline['ap50'], 4),
                  'per_class_ap50': {k: round(v, 4) for k, v in baseline['per_class_ap50'].items()},
                  'n_references': baseline['n_references']}, indent=2))

## 8. Bounded detection fine-tuning

This cell runs the real adaptation step in this runtime. It is gradient fine-tuning with the DETR loss YOLOS was trained with: each image's 100 detection tokens are matched one-to-one to its reference boxes by the Hungarian algorithm, and the matched tokens are trained with cross-entropy over the classes plus no-object (the no-object term is down-weighted by the config's `eos_coefficient` 0.1), an L1 box term and a generalised-IoU box term.

- **The early layers are frozen.** The patch embedding and encoder layers 0–7 keep their COCO values; the last 4 encoder layers, the detection tokens, the position embeddings, the final layer norm and both heads are trained. The cell prints the trainable and total parameter counts.
- **Schedule:** `EPOCHS` epochs of AdamW at learning rate `1e-4` (weight decay `1e-4`), batch size 4, gradient-norm clipping at 0.1, float32, seed `SEED`.

**Read the loss as optimisation evidence only.** A falling loss says the optimizer is fitting the training images; the held-out average precision in the next section is the task evidence.

In [ ]:
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}
FREEZE_EARLY_LAYERS = True  # @param {type:"boolean"}

run = adapter.finetune(
    train_records,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    freeze_early_layers=FREEZE_EARLY_LAYERS,
    progress=lambda row: print(f"epoch {row['epoch']}/{row['epochs']}  loss {row['loss']:.4f}"),
)
print(json.dumps({key: run[key] for key in ('freeze_early_layers', 'trainable_encoder_layers', 'trainable_parameters', 'total_parameters', 'epochs',
                                         'batch_size', 'learning_rate', 'optimizer', 'precision', 'device')}, indent=2))

## 9. Evaluate on the held-out split

`evaluate` re-runs on the same held-out images with the same thresholds as the baseline, so the two rows are comparable. `ap50` is average precision at IoU 0.50; `ap` averages AP over the ten IoU thresholds 0.50–0.95, so it also rewards tight boxes. Reading only `ap50` hides loose boxes; reading only `ap` hides whether objects were found at all. These are tutorial metrics from one pass over 10 synthetic images, with no dispersion estimate.

In [ ]:
adapted = adapter.evaluate(held_out)
print(f"{'metric':<8s} {'baseline':>10s} {'adapted':>10s} {'change':>10s}")
for key in ('ap', 'ap50', 'ap75'):
    print(f"{key:<8s} {baseline[key]:>10.4f} {adapted[key]:>10.4f} {adapted[key] - baseline[key]:>+10.4f}")
print('per-class AP50:', {k: round(v, 4) for k, v in adapted['per_class_ap50'].items()})

## 10. Inference on unseen images

Three new images come from a seed the dataset never used (`NEW_DATA_SEED = 99`). The adapted pipeline detects the new sign classes, and each reference box is compared with the best same-label detection by IoU.

In [ ]:
NEW_DATA_SEED = 99  # @param {type:"integer"}

new_records = sign_dataset(3, seed=NEW_DATA_SEED)
new_data_rows = []
for record in new_records:
    out = adapter.detect(record['image'], threshold=threshold)
    ious = []
    for box, label in zip(record['boxes'], record['labels'], strict=True):
        same_label = [d for d in out['detections'] if d['label'] == label]
        ious.append(round(max((box_iou(d['box'], box) for d in same_label), default=0.0), 3))
    row = {'id': record['id'], 'truth': record['labels'],
           'detections': [(d['label'], round(d['score'], 3)) for d in out['detections']], 'same_label_iou': ious}
    new_data_rows.append(row)
    print(json.dumps(row))

## 11. Adapter export, fresh reload, and equivalence check

`save_artifact` writes `outputs/yolos_adapter.safetensors`: every tensor the fine-tune could change, plus a metadata header naming the base model, its pinned revision, the base `model.safetensors` SHA-256, the class names and the frozen prefixes. The frozen patch embedding and the frozen encoder layers are left out because they equal the verified base snapshot.

`load_artifact` then builds a **fresh** pipeline from the verified base snapshot, loads the adapter tensors onto it, and refuses an adapter whose format, base identity, base digest or tensor set does not fit. The cell compares the reloaded detections with the in-memory model's on an unseen image, with a stated tolerance: loading succeeding is not the check, reproducing the detections is.

In [ ]:
artifact_path = OUTPUTS / 'yolos_adapter.safetensors'
descriptor = adapter.save_artifact(artifact_path, notes='YOLOS-Small sign adaptation tutorial adapter')
print(json.dumps(descriptor, indent=2))

reloaded = YolosDetectionPipeline.load_artifact(artifact_path, weights_dir=WEIGHTS_DIR)
print({'reloaded_source': reloaded.source, 'adapted': reloaded.adapted, 'class_names': list(reloaded.class_names)})

TOLERANCE = 1e-3
test_img = new_records[0]['image']
det_orig = adapter.detect(test_img, threshold=EVAL_DETECTION_THRESHOLD)['detections']
det_reloaded = reloaded.detect(test_img, threshold=EVAL_DETECTION_THRESHOLD)['detections']
assert len(det_orig) == len(det_reloaded)
for d1, d2 in zip(det_orig, det_reloaded, strict=True):
    assert d1['label'] == d2['label']
    assert np.allclose(d1['box'], d2['box'], atol=TOLERANCE)
    assert abs(d1['score'] - d2['score']) <= TOLERANCE
reload_check = {'detections_compared': len(det_orig), 'tolerance': TOLERANCE, 'equivalent': True}
print(reload_check)

## 12. Write machine-readable outputs and provenance

The cell writes:
- `outputs/yolos_detection_input_manifest.json`
- `outputs/yolos_detection_evaluation_report.json`
- `outputs/yolos_detection_result.json` (identity, runtime versions, device, dataset manifest, split, baseline and adapted metrics, fine-tuning configuration, new-data rows, adapter descriptor and reload check)
- `outputs/yolos_detection_detections.csv`
- `outputs/yolos_detection_annotated.png`
- `outputs/yolos_adapter.safetensors` (written in Section 11)

In [ ]:
import csv
from PIL import ImageDraw

with open(OUTPUTS / 'yolos_detection_input_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(input_manifest, f, indent=2)

with open(OUTPUTS / 'yolos_detection_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(coco_report, f, indent=2)

annotated = scene.copy()
draw = ImageDraw.Draw(annotated)
for det in coco_result['detections']:
    x0, y0, x1, y1 = det['box']
    draw.rectangle([x0, y0, x1, y1], outline='red', width=3)
    draw.text((x0 + 4, y0 + 4), f"{det['label']} {det['score']:.2f}", fill='red')
annotated.save(OUTPUTS / 'yolos_detection_annotated.png')

with open(OUTPUTS / 'yolos_detection_detections.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['image', 'rank', 'label', 'score', 'x0', 'y0', 'x1', 'y1'])
    for rank, d in enumerate(coco_result['detections']):
        writer.writerow(['tutorial-scene', rank, d['label'], f"{d['score']:.4f}", *(f"{v:.1f}" for v in d['box'])])
    for row, record in zip(new_data_rows, new_records, strict=True):
        for rank, d in enumerate(adapter.detect(record['image'], threshold=threshold)['detections']):
            writer.writerow([row['id'], rank, d['label'], f"{d['score']:.4f}", *(f"{v:.1f}" for v in d['box'])])

result_export = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__,
                'cuda': torch.cuda.is_available()},
    'device': pipe.device,
    'threshold': threshold,
    'coco_detections': coco_result['detections'],
    'degenerate_probes': degenerate,
    'adaptation': {
        'dataset': {k: v for k, v in dataset_manifest.items() if k != 'schema'},
        'dataset_seed': DATASET_SEED,
        'split': {'train': len(train_records), 'held_out': len(held_out), 'seed': SEED, 'holdout': HOLDOUT},
        'finetune': run,
        'baseline': baseline,
        'adapted': adapted,
        'new_data': new_data_rows,
        'artifact': descriptor,
        'reload_check': reload_check,
    },
}
with open(OUTPUTS / 'yolos_detection_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_export, f, indent=2, default=str)

for p in sorted(OUTPUTS.iterdir()):
    if p.is_file():
        print(f'  {p.name:<44s} {p.stat().st_size:>12,d} bytes')

## 13. Optional: Bring Your Own Data (BYOD)

Both branches are off by default, so `Run all` never stops here. Before you turn one on, read the contract:

- **Image branch** (`USE_BYOD_IMAGE`): one image file PIL can open, each side between `MIN_IMAGE_SIDE` (16) and `MAX_IMAGE_SIDE` (4096) px. It runs through `validate_inputs`, `detect` and `evaluation_report` — the report is `not-measurable`, because no reference boxes come with it.
- **Dataset branch** (`USE_BYOD_DATASET`): a directory with `annotations.json` (a list of `{'file', 'boxes', 'labels'}` objects, xyxy pixel boxes) and the images it names, at least 2 records and at most 5,000. File names must stay inside the directory. `BYOD_CLASS_NAMES` is a comma-separated vocabulary; leave it empty to use the sorted set of labels in the annotations. The branch runs the same validate → split → baseline → fine-tune → evaluate → export → reload stages as the sample.

Set `BYOD_IMAGE_PATH` or `BYOD_DATASET_DIR` to read from a mounted or local location; leave them empty on Colab to get an upload dialog instead. Uploaded files are written under `outputs/byod/` in this runtime and are not sent anywhere else. The first lines of the cell show the validator refusing two malformed inputs with messages that name the failed rule.

In [ ]:
USE_BYOD_IMAGE = False  # @param {type:"boolean"}
BYOD_IMAGE_PATH = ""  # @param {type:"string"}
USE_BYOD_DATASET = False  # @param {type:"boolean"}
BYOD_DATASET_DIR = ""  # @param {type:"string"}
BYOD_CLASS_NAMES = ""  # @param {type:"string"}

for desc, probe in (
    ('non-image object', lambda: validate_inputs('/not/an/image.png')),
    ('box outside the image', lambda: validate_dataset([{'image': blank_scene(), 'boxes': [[0, 0, 9999, 10]], 'labels': [SIGN_CLASSES[0]]}], SIGN_CLASSES)),
):
    try:
        probe()
    except (TypeError, ValueError) as exc:
        print(f'refused as expected: {desc} -> {type(exc).__name__}: {exc}')

BYOD_DIR = OUTPUTS / 'byod'

def _upload_into(target):
    from google.colab import files  # type: ignore[import-not-found]
    target.mkdir(parents=True, exist_ok=True)
    for name, data in files.upload().items():
        (target / Path(name).name).write_bytes(data)
    return target

if USE_BYOD_IMAGE:
    image_path = Path(BYOD_IMAGE_PATH) if BYOD_IMAGE_PATH else next(iter(sorted(_upload_into(BYOD_DIR / 'image').iterdir())))
    with Image.open(image_path) as handle:
        byod_image = handle.convert('RGB')
    print(validate_inputs(byod_image, threshold=threshold, names=[image_path.name])['verdict'])
    byod_result = pipe.detect(byod_image, threshold=threshold)
    for det in byod_result['detections'][:20]:
        print(f"{det['label']:>16s} {det['score']:.3f}  [{', '.join(f'{v:.0f}' for v in det['box'])}]")
    print(evaluation_report(byod_result, None, sample_kind='byod')['verdict'])
else:
    print('BYOD image branch is off; set USE_BYOD_IMAGE = True to run detection on your own image.')

if USE_BYOD_DATASET:
    dataset_dir = Path(BYOD_DATASET_DIR) if BYOD_DATASET_DIR else _upload_into(BYOD_DIR / 'dataset')
    byod_records = read_detection_records(dataset_dir)
    byod_names = [n.strip() for n in BYOD_CLASS_NAMES.split(',') if n.strip()] or sorted({l for r in byod_records for l in r['labels']})
    byod_manifest = validate_dataset(byod_records, byod_names, epochs=EPOCHS)
    print(json.dumps({k: v for k, v in byod_manifest.items() if k != 'schema'}, indent=2))
    byod_train, byod_held = split_dataset(byod_records, train_fraction=1.0 - HOLDOUT, seed=SEED)
    byod_pipe = YolosDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, class_names=byod_names, seed=SEED)
    byod_baseline = byod_pipe.evaluate(byod_held)
    byod_pipe.finetune(byod_train, epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, seed=SEED, freeze_early_layers=FREEZE_EARLY_LAYERS)
    byod_adapted = byod_pipe.evaluate(byod_held)
    print({'baseline_ap50': round(byod_baseline['ap50'], 4), 'adapted_ap50': round(byod_adapted['ap50'], 4)})
    byod_descriptor = byod_pipe.save_artifact(OUTPUTS / 'byod_yolos_adapter.safetensors', notes='BYOD adaptation adapter')
    YolosDetectionPipeline.load_artifact(OUTPUTS / 'byod_yolos_adapter.safetensors', weights_dir=WEIGHTS_DIR)
    print('BYOD adapter exported and reloaded:', byod_descriptor['sha256'][:16])
else:
    print('BYOD dataset branch is off; set USE_BYOD_DATASET = True to adapt YOLOS on your own labelled images.')

## Interpretation and limits

**What this notebook established, in this runtime.** The pinned `hustvl/yolos-small` snapshot was verified against a committed SHA-256 manifest before loading. The pretrained detector was run on a drawn scene and on two structure-free probes, and its boxes were compared with the drawn references by IoU. A three-class sign vocabulary that COCO does not contain was then adapted by replacing the last class-head layer, training the last encoder layers, the detection tokens and the heads with the early layers frozen, and scoring the held-out split with COCO-style average precision before and after. The adapter was exported, reloaded onto a fresh base model and checked against the in-memory detections.

**What a green run proves.** Successful execution proves that the recorded repository revision, the pinned dependency set and the pinned checkpoint together reproduce these stages in a fresh runtime, without the repository being cloned or installed and without any DIMER worker or service. It does **not** establish benchmark superiority, fitness for any deployment, or that the adapted model generalises beyond the synthetic images it was fitted to. The held-out AP is measured on 10 drawn images and carries no dispersion estimate. The scores are not calibrated probabilities.

**Reproducibility.** Seeds are form fields (`DATASET_SEED`, `SEED`, `NEW_DATA_SEED`), the run is float32 with no data augmentation, and the class head is initialised under `SEED`. GPU kernels are not forced to be deterministic, so repeated GPU runs can differ in the last digits of the loss and the scores.

**Try next.** Change `FREEZE_EARLY_LAYERS` to `False` and compare held-out AP and runtime, or change `EPOCHS` and watch where held-out AP stops improving. To transfer the workflow, point `BYOD_DATASET_DIR` at a small labelled set from your own domain.

## References

- Fang, Y., Liao, B., Wang, X., Fang, J., Qi, J., Wu, R., Niu, J. and Liu, W. (2021). *You Only Look at One Sequence: Rethinking Transformer in Vision through Object Detection.* [arXiv:2106.00666](https://arxiv.org/abs/2106.00666).
- Carion, N. et al. (2020). *End-to-End Object Detection with Transformers.* [arXiv:2005.12872](https://arxiv.org/abs/2005.12872) — the set-prediction loss.
- Upstream repository: [hustvl/YOLOS](https://github.com/hustvl/YOLOS) — MIT.
- Hugging Face checkpoint: [hustvl/yolos-small](https://huggingface.co/hustvl/yolos-small) — Apache-2.0.
- Lin, T.-Y. et al. (2014). *Microsoft COCO: Common Objects in Context.* [arXiv:1405.0312](https://arxiv.org/abs/1405.0312).
- Repository model card: https://github.com/kurtvalcorza/yolos-detection-pipeline/blob/main/MODEL_CARD.md
- [`kurtvalcorza/yolos-detection-pipeline`](https://github.com/kurtvalcorza/yolos-detection-pipeline) — source repository for this pipeline.